In [36]:
import json
import re

In [37]:
# buka data yang telah diekstrak
with open("extracted_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

In [38]:
# bersihkan price
def is_valid_price(item):
    price = item.get("price", "")
    if isinstance(price, int):
        return price > 0
    if isinstance(price, str):
        # longkapi string "Price Unavailable", kosong, dan teks invalid
        if not price or "Unavailable" in price or price.startswith("Price Unavailable"):
            return False
        # coba parsing string mata uang
        try:
            val = float(price.replace("$", ""))
            return val > 0
        except ValueError:
            return False
    return False

cleaned_price = [item for item in data if is_valid_price(item)]

# filter unknown product
cleaned_title = [item for item in cleaned_price if item.get("title") != "Unknown Product"]

# filter entri duplikat
seen = set()
unique = []
for item in cleaned_title:
    key = (
        item.get("title"),
        item.get("size"),
        item.get("gender")
    )
    if key not in seen:
        seen.add(key)
        unique.append(item)

with open("cleaned_data.json", "w", encoding="utf-8") as f:
    json.dump(unique, f, indent=2, ensure_ascii=False)

In [39]:
with open("cleaned_data.json", "r", encoding="utf-8") as f:
    data = json.load(f)

for entry in data:
    ### konversi price USD>IDR dari cleaned data
    price_str = entry["price"]
    price_num = float(price_str.replace("$", ""))
    entry["price"] = int(price_num*16000)

    ### konversi rating menjadi float
    rating_val = entry["rating"]
    # kalau sudah berupa float, skip
    if isinstance(rating_val, float):
        continue

    # kalau string, ekstrak floating point dari teks
    if isinstance(rating_val, str):
        match = re.search(r"⭐\s*([0-9]+(?:\.[0-9]+)?)\s*/", rating_val)
        if match:
            entry["rating"] = float(match.group(1))
        else:
            # jika tidak ada rating valid, jadikan 0.0
            entry["rating"] = None
    else:
        # jika tipe lain (None, int, etc), set ke None
        entry["rating"] = None

    ### bersihkan size
    size_val = entry["size"]
    if isinstance(size_val, str):
        # hapus prefix Size:
        entry["size"] = size_val.replace("Size: ", "").strip()

    ### bersihkan gender
    gender_val = entry["gender"]
    if isinstance(gender_val, str):
        # hapus prefix Gender:
        entry["gender"] = gender_val.replace("Gender: ", "").strip()

# simpan menjadi dataset baru
with open("transformed_data.json", "w", encoding="utf-8") as f:
    json.dump(data, f, indent=2, ensure_ascii=False)